# Connect to the Drive repo

Copying the repository in your drive. It will take some minutes if you do not have it yet:

In [ ]:
from pathlib import Path
import os, sys, subprocess

PUBLIC_ID = '1hpzEta1Ub_FlVQszrYXsis0Phirrjh81'
DEST_DIR = Path("/content/drive/My Drive/Finding_ES_LLM")
DRIVE_ROOT = Path("/content/drive")

# Mount

if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

# 2) check if the repo folder already exists in their Drive
if not DEST_DIR.exists() or not any(DEST_DIR.iterdir()):
    print("Repo not found in your Drive. Downloading …")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown>=5"])
    DEST_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [
        "gdown", f"https://drive.google.com/drive/folders/{PUBLIC_ID}",
        "--folder", "-O", str(DEST_DIR)
    ]
    subprocess.check_call(cmd)
else:
    print("Repo already present in your Drive -> skipping download")

# 3) put it on sys.path so imports work
sys.path.insert(0, str(DEST_DIR))
print("Repo ready at:", DEST_DIR)

# Setup and Imports

In these first two cells we run all the setup for the experiment. If you are met with a numpy dependency error just restart the notebook and it should be gone. Remember to change the huggingface secret with your own.

In [ ]:
from google.colab import userdata     # Imports secret tokens for logins
from huggingface_hub import login     # Huggingface for fast implementation of transformers

''' Change snippet and secrets below with your data '''

file_path = ('/content/drive/My Drive/Finding_ES_LLM/')
sys.path.append(file_path)
if not os.path.exists('/content/drive/My Drive'):
    from google.colab import drive
    drive.mount('/content/drive')

secrets = {
    'hugging': userdata.get('HF_TOKEN'),
    # 'wandb': userdata.get('wandb_api'),
    # 'nnsight': userdata.get('nnsight'),
    # 'openai': userdata.get('openai_api')
}

HF_TOKEN = secrets['hugging']
login(token=HF_TOKEN)

''' Change snippet and secrets above with your data '''

try:
    import transformer_lens as tlens
    import transformers as trans
    import openai
except:
    #  git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python
    # %pip install wandb -qU
    # %pip install -U bitsandbytes
    # %pip install -U accelerate
    %pip install -U transformers transformer_lens datasets einops jaxtyping
    %pip install openai
    import openai
    import transformer_lens as tlens
    import transformers as trans

"""
try:
    import sae_lens as slens
    from sae_lens import (
    SAE,
    ActivationsStore,
    HookedSAETransformer,
    LanguageModelSAERunnerConfig,
    SAEConfig,
    SAETrainingRunner,
    upload_saes_to_huggingface,
    )
    from sae_lens.toolkit.pretrained_saes_directory import get_pretrained_saes_directory
    from sae_vis import SaeVisConfig, SaeVisData, SaeVisLayoutConfig
except:
    # %pip install sae-lens
    #  git+https://github.com/callummcdougall/sae_vis.git@callum/v3
    %pip install openai>=1.56.2 nnsight
    %pip install --upgrade pydantic
    %pip install nnsight
"""

''' Navigate drive '''

import json
import pickle

''' For importing datasets '''

from datasets import load_dataset, Dataset

In [ ]:
# import nnsight

# import circuitsvis as cv

''' Tensor manipulation '''

import einops
from einops import einsum
import numpy as np
import torch as t                     # https://pytorch.org/docs/stable/torch.html
import torch.nn as nn                 # https://pytorch.org/docs/stable/nn.html
import torch.nn.functional as F       # https://pytorch.org/docs/stable/nn.functional.html

''' Utils '''

import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import gc
import seaborn as sns
import pickle

''' Should we need strong typing '''

from jaxtyping import Float, Int
from torch import Tensor
from typing import Callable, List, Tuple

''' For visualization of progress (in notebook) and training behavior (in wandb) '''

from tqdm import tqdm
# import wandb
# wandb.login(key=secrets['wandb'])
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

''' My things '''

from probes import SupervisedProbe
from visualization import *
from utils import *
from data_sets import *
from intervention import *
from subjprobs import *

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: alessandro-corona-m (alessandro-corona) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Model and Config

In [ ]:
# Inferred from exp 1, best layer for LR probes
# This is not the best layer for MMP probes, but it's a good enough approximation

best_layer = {'mmp': {
                'llama': 12,
                'llama_instruct': 14,
                'gemma': 25,
                'gemma_instruct': 30,
                'gpt-j': 22
                      },
              'lr': {
                'llama': 15,
                'llama_instruct': 13,
                'gemma': 27,
                'gemma_instruct': 28,
                'gpt-j': 13
                      }
              }

Here we init our model. The experiment is meant to be replicable with most families except gemma. Models from the Gemma family require some tweaking under the hood beacause of a different tokenizer.

We run the model in half precision: most models will be ok with an L4, but for gemma and llama-medium we will need A100's RAM

In [ ]:
# linear_layer best lr for residual stream on llama == ~1e-2/1e-3
# mlp best lr for residual stream on llama == ~1e-4
# mmp returns immediate results
# no batching is the fastest and works very well for cities.csv
# diminishing return after the 100th epoch for good activations

class ProbeConfig:
    def __init__(self):
        """ General """
        self.device = device
        self.batch_size_extractor = 128
        self.seed = 42
        """ Probe setup """
        self.probe_type = "mmp" # Options: linear_layer | linear | mlp | mmp
        self.supervision = "S"
        self.direction_type = 'mmp'  # Options: linear | logistic | mmp
        self.verbose = False
        self.with_std = True
        self.var_normalize = True
        self.control = False
        """ Specs """
        self.batch_size = -1
        self.nepochs = 50
        self.ntries = 1
        self.lr = 1e-3
        self.weight_decay = 0.0
        self.dropout = 0.0
        self.C = 1e6
        self.max_iter = 500
        self.test_size = 0.1
        self.log_accuracy_on_recursive = True
        self.patience = 10

probe_config = ProbeConfig()

MODEL = 'llama'

''' Dictionary for all of the models '''

mymodels = {
    'gemma': lambda: tlens.HookedTransformer.from_pretrained("gemma-2-9b", device=t.device('cpu')).half(),
    'gemma_instruct': lambda: tlens.HookedTransformer.from_pretrained("gemma-2-9b-it", device=t.device('cpu')).half(),
    'llama': lambda: tlens.HookedTransformer.from_pretrained("meta-llama/Llama-3.1-8B", device=t.device('cpu')).half(),
    'llama_instruct': lambda: tlens.HookedTransformer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", device=t.device('cpu')).half(),
    'gpt-j': lambda: tlens.HookedTransformer.from_pretrained("EleutherAI/gpt-j-6B", device=t.device('cpu')).half(),
}

model = mymodels[MODEL]()
model.to(device)

# TO DO: add SAE Hooked models

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Loaded pretrained model meta-llama/Llama-3.1-8B into HookedTransformer
Moving model to device:  cuda


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-31): 32 x TransformerBlock(
      (ln1): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): GroupedQueryAttention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): GatedMLP(
        (hook_pre): HookPoint()
        (hook_pre_linear): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_att

# Datasets

The datasets were built autonomously with the help of a GPT-4.1 mini API. We load the data from the folder and do some minimal preprocessing

In [ ]:
CUTOFF = 1500
file_path = ('/content/drive/My Drive/Finding_ES_LLM/')
folder = 'datasets_diy/'

# Logical - easy
with open(f'{file_path}{folder}curated_dataset_full.pkl', 'rb') as file:
    curated_dataset = pickle.load(file)
with open(f'{file_path}{folder}neg_dataset.pkl', 'rb') as file:
    negated = pickle.load(file)
with open(f'{file_path}{folder}disj_dataset.pkl', 'rb') as file:
    disjunction = pickle.load(file)
with open(f'{file_path}{folder}conj_dataset.pkl', 'rb') as file:
    conjunction = pickle.load(file)
# Logical - inference
with open(f'{file_path}{folder}larger_than_inference.pkl', 'rb') as file:
    lt_inference = pickle.load(file)
with open(f'{file_path}{folder}smaller_than_inference.pkl', 'rb') as file:
    st_inference = pickle.load(file)
with open(f'{file_path}{folder}cities_inference.pkl', 'rb') as file:
    cities_inference = pickle.load(file)
with open(f'{file_path}{folder}companies_inference.pkl', 'rb') as file:
    companies_inference = pickle.load(file)
with open(f'{file_path}{folder}commonclaim_inference.pkl', 'rb') as file:
    cc_inference = pickle.load(file)
with open(f'{file_path}{folder}counterfact_inference.pkl', 'rb') as file:
    cf_inference = pickle.load(file)
# Other datasets
with open(f'{file_path}{folder}mymulan.pkl', 'rb') as file:
    mulan = pickle.load(file)
with open(f'{file_path}{folder}tqa_curated.pkl', 'rb') as file:
    tqa = pickle.load(file)
with open(f'{file_path}{folder}likely.pkl', 'rb') as file:
    likely = pickle.load(file)
with open(f'{file_path}{folder}entailment_new.pkl', 'rb') as file:
    entailment_new = pickle.load(file)

''' Data cleanup and division '''

# Clean 'negated'

new_column_for_neg = curated_dataset[curated_dataset['filename'].isin(['common_claim_true_false.csv', 'companies_true_false.csv', 'counterfact_true_false.csv'])]
new_column_for_neg['filename'].unique()
negated = negated.rename(columns={'statement':'new_statement'})
negated['statement'] = new_column_for_neg['statement']
negated = negated[['statement', 'new_statement', 'label', 'filename']]
negated['label'] = 1 - negated['label']
negated['neg_label'] = 1 - negated['label']

# Mulan division

mulan_mutable = mulan[mulan['type'] == 'mutable']
mulan_immutable = mulan[mulan['type'] == 'immutable']
mulan_mutable = stratified_sample(mulan_mutable, 'relation', CUTOFF)
mulan_immutable = stratified_sample(mulan_immutable, 'relation', CUTOFF)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/Finding_ES_LLM/datasets_diy/curated_datasetss_full.pkl'

Here we get the data split for the experiment. Each task has a different data split (accessing different dataset having different paraphrases).

In [ ]:
X_hop = None    # Discriminates inference task from others

datasplit = get_data_split('negation', curated_dataset, probe_config=probe_config, other_dataset=entailment_new, cutoff=CUTOFF)

'''
datasplit is a tuple containing all relevant data for the coherence experiment
it returns: X_clean_train, y_clean_train, X_clean_test_base, X_clean_test_paraph, y_clean_test_label
in case of inference task (we need another set of statements): X_clean_train, y_clean_train, X_clean_test_base, X_clean_test_paraph, X_clean_test_hop, y_clean_test_label
test labels are helpers for activation extractor, they are not important for our metrics
'''

X_clean_train, y_clean_train, df = datasplit

## Training Loop

Here we train the LR Ensemble and the MMP probe for subsequent subjprob extraction

# Probe training

In [ ]:
# == Resid

model.reset_hooks()
resid_extractor = ActivationExtractor(model=model, data=X_clean_train, labels=y_clean_train, device=device, half=True,
                                      batch_size=probe_config.batch_size_extractor)
resid_extractor.set_hooks(
                          [best_layer[probe_config.probe_type][MODEL]],
                          [tlens.utils.get_act_name('resid_post')], attn=False)
resid_activations, resid_labels = resid_extractor.process() # Get

model.to(t.device('cpu'))
gc.collect()
t.cuda.empty_cache()

Moving model to device:  cuda


Processing: 100%|██████████| 86/86 [02:34<00:00,  1.80s/it]


Moving model to device:  cpu


### LR


In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*To copy construct from a tensor.*")            # Ignores an annoying warning from sklearn

probe_config.verbose=False
probe_config.control=False
probe_config.lr=0.0009
probe_config.nepochs=1000
probe_config.batch_size=512
probe_config.dropout=0.2

dataset = einops.rearrange(next(iter(resid_activations.values())), 'n b d -> (n b) d')
gold = einops.rearrange(resid_labels, 'n b -> (n b)')
X_train, X_test, y_train, y_test = train_test_split(dataset, gold, test_size=probe_config.test_size, random_state=probe_config.seed)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

master_probe, directions = form_master_probe(probe_config, X_train, y_train, X_test, y_test, n=10)
master_probe.to(device)

Epochs:  26%|██▌       | 255/1000 [00:30<01:27,  8.47it/s]


Early stopping triggered at epoch 256
Run 1/10 accuracy: 0.8837


Epochs:  23%|██▎       | 228/1000 [00:26<01:28,  8.68it/s]


Early stopping triggered at epoch 229
Run 2/10 accuracy: 0.8901


Epochs:  29%|██▉       | 293/1000 [00:34<01:22,  8.55it/s]


...

Epochs:  33%|███▎      | 331/1000 [00:39<01:18,  8.47it/s]


Early stopping triggered at epoch 332
Run 9/10 accuracy: 0.8865


Epochs:  24%|██▍       | 244/1000 [00:28<01:28,  8.53it/s]

Early stopping triggered at epoch 245
Run 10/10 accuracy: 0.8792


Sequential(
  (0): FixedLinear()
  (1): Sigmoid()
)

### MMP

In [ ]:
# Mass-mean Probe

with t.autocast('cuda'):
  X_train = X_train.astype(np.float32)
  X_test = X_test.astype(np.float32)
  probe_config.var_normalize = False
  mm_probe = SupervisedProbe(x_train=X_train, labels_train=y_train,
                          x_test=X_test, labels_test=y_test,
                          probe_cfg=probe_config)
  mm_probe.repeated_train()
  mm = mm_probe.get_direction()
  mm_acc = mm_probe.get_acc()

0.846503178928247


In [ ]:
# cleanup resid activations and labels from device if needed

del resid_activations
del resid_labels
gc.collect()
t.cuda.empty_cache()

# Tests - Interp

Here we test the different estimators on a single task. From now on, the relevant code for the task has to be uncommented. The notebook is set for a negation task

## Conj/Disj/Neg/CrossDataset

Here df_a and df_b will be the same df (for logical paraphrases) and the two confronted df for cross-dataset.

For example:

* If running a test on mulan, df_a will contain immutable facts and df_b will contain mutable facts.
* If running a test on TQA, df_a will contain our base dataset and df_b will contain statements extracted from TQA

In [ ]:
# Extract representations for test set

X_hop = None

df_a = df[0].iloc[:-(len(df[0]) % probe_config.batch_size_extractor)]
df_b = df[1].iloc[:-(len(df[1]) % probe_config.batch_size_extractor)]

# For cross-dataset task:

'''
X_base = list(df_a['statement'])
X_paraph = list(df_b['statement'])
labels = list(df_a['label'])
'''

# for logical non-inference tasks:

X_base = list(df_a['statement'])
X_paraph = list(df_a['new_statement'])
labels = list(df_a['label'])
df = df_a

# for inference task:
'''
X_base = list(df_a['statement'])
X_paraph = list(df_a['new_statement'])
X_hop = list(df_a['hop_statement'])
labels = list(df_a['label'])
'''

"\nX_base = list(df_a['statement'])\nX_paraph = list(df_a['new_statement'])\nX_hop = list(df_a['hop_statement'])\nlabels = list(df_a['label'])\n"

Here we collect the activations for interp estimators

In [ ]:
''' Set up X_base and X_paraph '''

model.reset_hooks()
base_extractor = ActivationExtractor(model=model, data=X_base, labels=labels, device=device, half=True,
                                      batch_size=probe_config.batch_size_extractor)
base_extractor.set_hooks(
                          [best_layer[probe_config.probe_type][MODEL]],
                          [tlens.utils.get_act_name('resid_post')], attn=False) # for instance
base_activations, base_labels = base_extractor.process() # Get
# model.reset_hooks()
paraph_extractor = ActivationExtractor(model=model, data=X_paraph, labels=labels, device=device, half=True,
                                      batch_size=probe_config.batch_size_extractor)
paraph_extractor.set_hooks(
                          [best_layer[probe_config.probe_type][MODEL]],
                          [tlens.utils.get_act_name('resid_post')], attn=False) # for instance
paraph_activations, paraph_labels = paraph_extractor.process() # Get

base_activations_values = base_activations[next(iter(base_activations))]
base_activations_values = einops.rearrange(base_activations_values, 'n b d -> (n b) d')
base_activations_values = scaler.transform(base_activations_values)
paraph_activations_values = paraph_activations[next(iter(paraph_activations))]
paraph_activations_values = einops.rearrange(paraph_activations_values, 'n b d -> (n b) d')
paraph_activations_values = scaler.transform(paraph_activations_values)

if X_hop is not None:   # Get data for entailment
  model.reset_hooks()
  hop_extractor = ActivationExtractor(model=model, data=X_hop, labels=y_test, device=device, half=True,
                                        batch_size=probe_config.batch_size_extractor)
  hop_extractor.set_hooks(
                            [best_layer[probe_config.probe_type][MODEL]],
                            [tlens.utils.get_act_name('resid_post')], attn=False) # for instance
  hop_activations, hop_labels = hop_extractor.process() # Get

  hop_activations_values = hop_activations[next(iter(hop_activations))]
  hop_activations_values = einops.rearrange(hop_activations_values, 'n b d -> (n b) d')
  hop_activations_values = scaler.transform(hop_activations_values)

model.to(t.device('cpu'))
gc.collect()
t.cuda.empty_cache()

Moving model to device:  cuda


Processing: 100%|██████████| 14/14 [00:25<00:00,  1.81s/it]


Moving model to device:  cuda


Processing: 100%|██████████| 14/14 [00:45<00:00,  3.27s/it]


Moving model to device:  cpu


## Interp

### LR

In [ ]:
master_base_probabilities = master_probe(t.tensor(base_activations_values, device=device, dtype=t.float32))
master_paraph_probabilities = master_probe(t.tensor(paraph_activations_values, device=device, dtype=t.float32))

''' Logical paraphrases '''

df['master_label_base'] = master_base_probabilities.cpu().numpy() > 0.5
df['master_label_paraph'] = master_paraph_probabilities.cpu().numpy() > 0.5
df['master_proba_base'] = master_base_probabilities.cpu().numpy()
df['master_proba_paraph'] = master_paraph_probabilities.cpu().numpy()
if X_hop is not None:
  master_hop_probabilities = master_probe(t.tensor(hop_activations_values, device=device, dtype=t.float32))
  df['master_label_hop'] = master_hop_probabilities.cpu().numpy() > 0.5
  df['master_proba_hop'] = master_hop_probabilities.cpu().numpy()

''' Cross-dataset '''

# df_a['master_label_base'] = master_base_probabilities.cpu().numpy() > 0.5
# df_b['master_label_paraph'] = master_paraph_probabilities.cpu().numpy() > 0.5
# df_a['master_proba_base'] = master_base_probabilities.cpu().numpy()
# df_b['master_proba_paraph'] = master_paraph_probabilities.cpu().numpy()

/content/drive/My Drive/Finding_ES_LLM/subjprobs.py:106: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4421.)
  return t.matmul(x, self.weight.T)
/tmp/ipython-input-3291667168.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['master_label_base'] = master_base_probabilities.cpu().numpy() > 0.5
/tmp/ipython-input-3291667168.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .lo

### MMP

In [ ]:
with t.autocast('cuda'):

  ''' Logical Paraphrases '''

  mm_base_probabilities = mm_probe.probe(t.tensor(base_activations_values, device=device, dtype=t.float32))
  mm_paraph_probabilities = mm_probe.probe(t.tensor(paraph_activations_values, device=device, dtype=t.float32))
  df['mm_label_base'] = mm_base_probabilities.detach().cpu().numpy() > 0.5
  df['mm_label_paraph'] = mm_paraph_probabilities.detach().cpu().numpy() > 0.5
  df['mm_proba_base'] = mm_base_probabilities.detach().cpu().numpy()
  df['mm_proba_paraph'] = mm_paraph_probabilities.detach().cpu().numpy()
  if X_hop is not None:
    mm_hop_probabilities = mm_probe.probe(t.tensor(hop_activations_values, device=device, dtype=t.float32))
    df['mm_label_hop'] = master_hop_probabilities.detach().cpu().numpy() > 0.5
    df['mm_proba_hop'] = master_hop_probabilities.detach().cpu().numpy()

  ''' Cross-dataset '''

  # df_a['mm_label_base'] = mm_base_probabilities.detach().cpu().numpy() > 0.5
  # df_b['mm_label_paraph'] = mm_paraph_probabilities.detach().cpu().numpy() > 0.5
  # df_a['mm_proba_base'] = mm_base_probabilities.detach().cpu().numpy()
  # df_b['mm_proba_paraph'] = mm_paraph_probabilities.detach().cpu().numpy()

/tmp/ipython-input-354417761.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['mm_label_base'] = mm_base_probabilities.detach().cpu().numpy() > 0.5
/tmp/ipython-input-354417761.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['mm_label_paraph'] = mm_paraph_probabilities.detach().cpu().numpy() > 0.5
/tmp/ipython-input-354417761.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats 

## Non-interp

Behavioral tests take a lot more time and resources than the probes since the model has to generate the logits/answers. Depending on the compute it may take up to ~3 hrs for self-reporting (inference task) and ~15 mins for logit estimates

### Self-reporting

In [ ]:
model.reset_hooks()
model.to(device)
def embed_to_half_hook(tensor, hook):
    return tensor.half()

model.add_hook("hook_embed", embed_to_half_hook)

import gc

context = "I am a fact-checking AI. For each statement, I determine whether it is True or False and rate my confidence in that judgment on a scale from 0 to 1."
shots = [
    'Statement: Paris is the capital of France.\nAnswer: True\nConfidence: 0.95',
    'Statement: The largest bear in the world is currently in Italy.\nAnswer: False\nConfidence: 0.75',
    'Statement: Milan is the capital of Italy.\nAnswer: False\nConfidence: 0.85',
    'Statement: Pythagoras was originally from Turkey.\nAnswer: False\nConfidence: 0.70'
]

tqdm.pandas()


Moving model to device:  cuda


In [ ]:
''' Logical '''

df[['self_label_base', 'self_proba_base']] = df['statement'].progress_apply(lambda v: self_evaluate_row(v, model=model, context=context, shots=shots))
df[['self_label_paraph', 'self_proba_paraph']] = df['new_statement'].progress_apply(lambda v: self_evaluate_row(v, model=model, context=context, shots=shots))
if X_hop is not None:
  df[['self_label_hop', 'self_proba_hop']] = df['hop_statement'].progress_apply(lambda v: self_evaluate_row(v, model=model, context=context, shots=shots))

backup_df = df
df = df.dropna()

''' Cross-dataset '''

# df_a[['self_label_base', 'self_proba_base']] = df_a['statement'].progress_apply(lambda v: self_evaluate_row(v, model=model, context=context, shots=shots))
# df_b[['self_label_paraph', 'self_proba_paraph']] = df_b['statement'].progress_apply(lambda v: self_evaluate_row(v, model=model, context=context, shots=shots))

100%|██████████| 1792/1792 [49:35<00:00,  1.66s/it]
/tmp/ipython-input-3472476557.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[['self_label_base', 'self_proba_base']] = df['statement'].progress_apply(lambda v: self_evaluate_row(v, model=model, context=context, shots=shots))
/tmp/ipython-input-3472476557.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[['self_label_base', 'self_proba_base']] = df['statement'].progress_apply(lambda v: self_evaluate_row(v, model=model, context=context, shots=sh

' Cross-dataset '

In [ ]:
df.head()

,statement,label,filename,new_statement,hop_statement,master_label_base,master_label_paraph,master_proba_base,master_proba_paraph,mm_label_base,mm_label_paraph,mm_proba_base,mm_proba_paraph,self_label_base,self_proba_base,self_label_paraph,self_proba_paraph
902,"Every day, the average person walks the equiva...",0,common_claim_true_false.csv,"It is both the case that Every day, the averag...",Walking three times around the equator is equi...,False,False,0.249407,0.007160,False,False,0.477783,0.000843,False,0.35,True,0.90
432,Interstate 255 is within the metropolitan area...,0,counterfact_true_false.csv,It is both the case that Interstate 255 is wit...,Vienna is within the metropolitan area of Vienna.,False,False,0.068918,0.013336,False,False,0.071472,0.352051,False,0.35,False,0.20
3610,The earth is over four and a half billion year...,1,common_claim_true_false.csv,It is both the case that The earth is over fou...,Being over four and a half billion years old m...,True,True,0.981192,0.805157,True,True,0.999023,0.989746,True,0.95,True,0.95
454,Chameleons can change color to match passing w...,0,common_claim_true_false.csv,It is both the case that Chameleons can change...,White cumulus clouds are a type of passing clo...,False,False,0.204407,0.005698,True,False,0.984375,0.015839,True,0.90,True,0.90
324,Giorgio Albertazzi writes in British English.,0,counterfact_true_false.csv,It is both the case that Giorgio Albertazzi wr...,British English is a variety of English.,False,False,0.068809,0.015711,False,False,0.188721,0.027847,True,0.90,True,0.95


### Logit-based

In [ ]:
# Apply it to the DataFrame
df[['logit_label_base', 'logit_proba_base']] = df['statement'].progress_apply(lambda v: logit_evaluate_row(v, model=model, shots=shots))
df[['logit_label_paraph', 'logit_proba_paraph']] = df['new_statement'].progress_apply(lambda v: logit_evaluate_row(v, model=model, shots=shots)) # Fix with the correct column
if X_hop is not None:
  df[['logit_label_hop', 'logit_proba_hop']] = df['hop_statement'].progress_apply(lambda v: logit_evaluate_row(v, model=model, shots=shots))

''' Cross-dataset '''

# df_a[['self_label_base', 'self_proba_base']] = df_a['statement'].progress_apply(lambda v: logit_evaluate_row(v, model=model, shots=shots)))
# df_b[['self_label_base', 'self_proba_base']] = df_b['statement'].progress_apply(lambda v: logit_evaluate_row(v, model=model, shots=shots))

100%|██████████| 1792/1792 [03:31<00:00,  8.49it/s]


' Cross-dataset '

## We have full probas. Let us turn to evaluation

In [ ]:
# Random baselines of various sorts (esp. for negation task)

random_base = t.rand(len(df))
random_hop = t.rand(len(df))
random_base = np.array(df['logit_proba_base'])
np.random.shuffle(random_base)
random_hop = np.array(df['logit_proba_paraph'])
np.random.shuffle(random_base)

In [ ]:
''' Set up judges '''

judge_negation = JudgeCoherence(logic='neg')
judge_negation.set_metric(judge_negation.rmse_metric)
judge_disjunction = JudgeCoherence(logic='disj')
judge_disjunction.set_metric(judge_disjunction.less_than_perc)
judge_conjunction = JudgeCoherence(logic='conj')
judge_disjunction.set_metric(judge_conjunction.less_than_perc)
judge_ent = JudgeCoherence(logic='ent')
judge_ent.set_metric(judge_ent.mae_metric_clamp)
judge_ent_ = JudgeCoherence(logic='ent*')
judge_ent.set_metric(judge_ent.mae_metric_clamp)
judge_cross = JudgeCoherence(logic='datasets')
judge_ent.set_metric(judge_ent.avg_conf_diff)

''' neg '''

neg = {
    'master': judge_negation.judge([t.tensor(list(df['master_proba_base'])), t.tensor(list(df['master_proba_paraph']))]),
    'mm': judge_negation.judge([t.tensor(list(df['mm_proba_base'])), t.tensor(list(df['mm_proba_paraph']))]),
    'self': judge_negation.judge([t.tensor(list(df['self_proba_base'])), t.tensor(list(df['self_proba_paraph']))]),
    'logit': judge_negation.judge([t.tensor(list(df['logit_proba_base'])), t.tensor(list(df['logit_proba_paraph']))])
}

for key, value in neg.items():
    print(f"Performance on negation - {key}: {value}")

''' disj '''

# disj = {
#     'master': judge_disjunction.judge([t.tensor(list(df['master_proba_base'])), t.tensor(list(df['master_proba_paraph']))]),
#     'mm': judge_disjunction.judge([t.tensor(list(df['mm_proba_base'])), t.tensor(list(df['mm_proba_paraph']))]),
#     'self': judge_disjunction.judge([t.tensor(list(df['self_proba_base'])), t.tensor(list(df['self_proba_paraph']))]),
#     'logit': judge_disjunction.judge([t.tensor(list(df['logit_proba_base'])), t.tensor(list(df['logit_proba_paraph']))])
# }

# for key, value in disj.items():
#     print(f"Performance on disjunction - {key}: {value}")

''' conj '''

# conj = {
#     'master': judge_disjunction.judge([t.tensor(list(df['master_proba_paraph'])), t.tensor(list(df['master_proba_base']))]),
#     'mm': judge_disjunction.judge([t.tensor(list(df['mm_proba_paraph'])), t.tensor(list(df['mm_proba_base']))]),
#     'self': judge_disjunction.judge([t.tensor(list(df['self_proba_paraph'])), t.tensor(list(df['self_proba_base']))]),
#     'logit': judge_disjunction.judge([t.tensor(list(df['logit_proba_paraph'])), t.tensor(list(df['logit_proba_base']))])
# }

# for key, value in disj.items():
#     print(f"Performance on conjunction - {key}: {value}")

''' inference '''

# ent = {
#     'master': judge_ent.judge([t.tensor(list(df['master_proba_paraph'])), t.tensor(list(df['master_proba_base'])), t.tensor(list(df['master_proba_hop']))]),
#     'mm': judge_ent.judge([t.tensor(list(df['mm_proba_paraph'])), t.tensor(list(df['mm_proba_base'])), t.tensor(list(df['mm_proba_hop']))]),
#     'self': judge_ent.judge([t.tensor(list(df['self_proba_paraph'])), t.tensor(list(df['self_proba_base'])), t.tensor(list(df['self_proba_hop']))]),
#     'logit': judge_ent.judge([t.tensor(list(df['logit_proba_paraph'])), t.tensor(list(df['logit_proba_base'])), t.tensor(list(df['logit_proba_hop']))])
# }

# for key, value in ent.items():
#     print(f"Performance on inference - {key}: {value}")

# ent_ = {
#     'master': judge_ent_.judge([t.tensor(list(df['master_proba_paraph'])), t.tensor(list(df['master_proba_base'])), t.tensor(list(df['master_proba_hop']))]),
#     'mm': judge_ent_.judge([t.tensor(list(df['mm_proba_paraph'])), t.tensor(list(df['mm_proba_base'])), t.tensor(list(df['mm_proba_hop']))]),
#     'self': judge_ent_.judge([t.tensor(list(df['self_proba_paraph'])), t.tensor(list(df['self_proba_base'])), t.tensor(list(df['self_proba_hop']))]),
#     'logit': judge_ent_.judge([t.tensor(list(df['logit_proba_paraph'])), t.tensor(list(df['logit_proba_base'])), t.tensor(list(df['logit_proba_hop']))])
# }

# for key, value in ent_.items():
#     print(f"Performance on inference - {key}: {value}")

''' Cross-dataset '''

# cross = {
#     'master': judge_cross.judge([t.tensor(list(df_a['master_proba_base'])), t.tensor(list(df_b['master_proba_paraph']))]),
#     'mm': judge_cross.judge([t.tensor(list(df_a['mm_proba_base'])), t.tensor(list(df_b['mm_proba_paraph']))]),
#     'self': judge_cross.judge([t.tensor(list(df_a['self_proba_base'])), t.tensor(list(df_b['self_proba_paraph']))]),
#     'logit': judge_cross.judge([t.tensor(list(df_a['logit_proba_base'])), t.tensor(list(df_b['logit_proba_paraph']))])
# }

# for key, value in cross.items():
#     print(f"Performance on cross-dataset - {key}: {value}")

Performance on negation - master: 0.7012952566146851
Performance on negation - mm: 0.6774092316627502
Performance on negation - self: 0.6949184536933899
Performance on negation - logit: 0.7882452011108398


' Cross-dataset '

# Self-Consistency Study

Here we run the self-consistency study

In [ ]:
df_backup = df
df = df.dropna()

In [ ]:
def pcorrect(proba):

  return max(proba, 1-proba)

def preddy(stringz):
    if stringz == 'True':
        return 1
    elif stringz == 'False':
        return 0
    else:
        return 0

y_pred = df['logit_label_base'].apply(preddy)
y_true = df['label']
conf_master = df['master_proba_base']
conf_mm = df['mm_proba_base'].astype(float)
conf_logit = df['logit_proba_base']
conf_self = df['self_proba_base']
corrects = (1 == y_pred).astype(int)
cmapz = {
    'LR': 'Blues',
    'MMP': 'Greens',
    'Logit': 'YlOrBr',
    'Self': 'Purples'
}

types = {
    'LR': conf_master,
    'MMP': conf_mm,
    'Logit': conf_logit,
    'Self': conf_self
}
TYPE = 'LR'

In [ ]:
bin_stats, bin_edges = calibration_curve(types[TYPE], y_pred, n_bins=10, quantile_binning=True)

fig, ax = plt.subplots(figsize=(8, 6))

# Plot the reliability diagram
plot_reliability_diagram(
    bin_stats=bin_stats,
    bin_edges=bin_edges,
    ax=ax,
    title=f"Self-Consistency ({TYPE})",
    cmap=cmapz[TYPE]  # or any other colormap like 'coolwarm', 'Blues', etc.
)

plt.tight_layout()
plt.show()

### Legacy tests

These cells contain some legacy tests. We mostly have a cosine similarity matrix for LR thetas and a following orthogonal probing experiment, testing the leakiness of a LR-style model and MMP

In [ ]:
cossims = []

for direction in directions:

    cossims.append(cosine_similarity(direction.reshape(1, -1), mm.reshape(1, -1))[0])

print(np.array(cossims).mean())

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def plot_cosine_similarity_heatmap(sim_matrix, labels=None):
    plt.figure(figsize=(10, 8))

    ax = sns.heatmap(
        sim_matrix,
        xticklabels=labels,
        yticklabels=labels,
        cmap="mako",
        annot=True,
        fmt=".3f",
        vmin=0.9,
        vmax=1.000,
        linewidths=0.5,
        linecolor='white',
        square=True,
        cbar_kws={"shrink": 1.00}
    )

    ax.set_title("Pairwise Cosine Similarity LRs", fontsize=16, weight='bold', pad=20)
    ax.set_xlabel("Direction", fontsize=12)
    ax.set_ylabel("Direction", fontsize=12)

    # Rotate x-axis labels for better readability
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(fontsize=10)

    plt.tight_layout()
    plt.show()
def cosine_matrix(vectors):

    similarity_matrix = cosine_similarity(vectors)
    labels = [f'D{i}' for i in range(len(vectors))]
    plot_cosine_similarity_heatmap(similarity_matrix, labels)
    return similarity_matrix

similarity_matrix = cosine_matrix(directions)

In [ ]:
accuracies_simple = orthogonal_probing(probe_config, X_train, y_train, X_test, y_test, n=20, fix_direction=None)
accuracies_mm = orthogonal_probing(probe_config, X_train, y_train, X_test, y_test, n=19, fix_direction=mm)
accuracies_mm.insert(0, mm_acc)

master_probe_direction = next(param for param in master_probe.parameters()).cpu().numpy()
accuracies_master = orthogonal_probing(probe_config, X_train, y_train, X_test, y_test, n=19, fix_direction=master_probe_direction)
accuracies_master.insert(0, accuracy).cpu().numpy() # accuracy = accuracy for the master probe

accuracies = {'Mass-mean': accuracies_mm, 'LR': accuracies_simple, 'Average LR': accuracies_master}

In [ ]:
sns.set(style="whitegrid", context="talk", palette="deep")
plt.figure(figsize=(8, 6))

for key, value in accuracies.items():
    plt.plot(value, label=key, linewidth=2, marker='o', markersize=7, alpha=0.9)

plt.xlabel('Probing Iteration', fontsize=14)
plt.ylabel('Probe Accuracy', fontsize=14)
plt.title('Orthogonal Probing', fontsize=16, weight='bold')
plt.axhspan(0.45, 0.55, color='grey', alpha=0.15, zorder=0)
plt.xticks(fontsize=12, ticks=range(20))
plt.yticks(fontsize=12)
plt.legend(title="Probe", fontsize=11, title_fontsize=12, loc='best')
plt.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.8)

plt.tight_layout()
plt.show()
